# Датасет: E-Commerce Sales and Customer Analytics
**Пепеляев Иван, 3823Б1ПМоп2**

Источник: [Kaggle](https://www.kaggle.com/datasets/datascikhan/e-commerce-sales-and-customer-analytics), файл `ecommerce_sales_customer_analytics_150k.csv`.

**Задача:** классификация — предсказать тональность отзыва (`review_sentiment`: Positive, Neutral, Negative) по характеристикам заказа.

In [15]:
import pandas as pd
import numpy as np

# Загружаем датасет (low_memory=False убирает предупреждение о смешанных типах)
df = pd.read_csv('ecommerce_sales_customer_analytics_150k.csv', low_memory=False)
print(df.shape)
df.head()

(138116, 46)


,order_id,order_date,order_time,order_status,sales_channel,customer_id,customer_name,customer_age,gender,customer_segment,...,discount_amount,tax_amount,shipping_cost,net_sales,product_cost,profit,profit_margin_percentage,customer_lifetime_value,is_repeat_customer,customer_order_count
0,ORD-301242,2023-11-06,16:37:47,Completed,Mobile App,CUST-003102,Jasmine Ryan,55,Male,Consumer,...,477.89,61.07,12.03,945.40,588.33,345.04,36.50,12459.68,True,11
1,ORD-773460,2025-12-24,01:22:36,Completed,Website,CUST-003124,Scott Chase,26,Male,Premium,...,1456.06,320.83,9.00,2018.41,2031.88,-22.47,-1.11,13032.48,True,11
2,ORD-449374,2021-07-05,14:24:21,Completed,Mobile App,CUST-012496,Marc Wheeler,69,Female,Premium,...,97.30,29.79,13.05,468.35,257.73,197.57,42.18,6159.44,True,6
3,ORD-567636,2023-01-21,07:20:26,Completed,Social Media,CUST-023928,Jennifer Smith,65,Male,Consumer,...,53.34,34.39,20.85,546.52,277.84,247.83,45.35,5638.30,True,7
4,ORD-820028,2022-05-13,09:46:21,Completed,Social Media,CUST-012730,Jessica Wang,34,Female,Consumer,...,133.63,421.35,23.03,2785.27,1231.88,1530.36,54.94,13240.10,True,8


## 1. Отбор признаков
Убираем служебные столбцы (id, имя, даты), целевую переменную и столбцы, которые «подсказывают» ответ (утечка).


In [16]:
target = "review_sentiment"

service = ["order_id", "customer_id", "customer_name", "order_date", "order_time"]
leak    = ["return_reason", "return_status", "customer_rating"]
drop_cols = set(service + leak + [target])

N = len(df)
d = len(df.columns)
features = [c for c in df.columns if c not in drop_cols]

print("N (объектов):", N)
print("d (признаков всего):", d)
print("Признаков после удаления служебных и утечек:", len(features))

N (объектов): 138116
d (признаков всего): 46
Признаков после удаления служебных и утечек: 37


## 2. Типы признаков

In [17]:
# Автоматически определяем типы через pandas - никакого ручного списка!
num_cols_all = [c for c in df.select_dtypes(include=[np.number]).columns if c not in drop_cols]
obj_cols_all = [c for c in df.select_dtypes(include=['object']).columns if c not in drop_cols]
bool_cols    = [c for c in df.select_dtypes(include=['bool']).columns if c not in drop_cols]

# Разделяем строковые на текстовые и категориальные
# Текстовый = более 1000 уникальных значений или средняя длина строки > 50 символов
text_cols = [c for c in obj_cols_all
             if df[c].nunique() > 1000 or df[c].astype(str).str.len().mean() > 50]
cat_cols  = [c for c in obj_cols_all if c not in text_cols]

num_cols   = num_cols_all
other_cols = bool_cols

print("Числовых:", len(num_cols))
print("Категориальных:", len(cat_cols))
print("Текстовых:", len(text_cols), text_cols)
print("Других типов:", len(other_cols), other_cols)
print()
print("Пример текстового признака:")
print(df[text_cols[0]].dropna().iloc[0][:300])

Числовых: 17
Категориальных: 18
Текстовых: 1 ['customer_city']
Других типов: 1 ['is_repeat_customer']

Пример текстового признака:
Lake Williamberg


## 3. Целевая переменная и пропуски

In [18]:
# Классы считаем только по строкам, где целевая переменная известна
df_target = df.dropna(subset=[target])
K = df_target[target].nunique()
class_counts = df_target[target].value_counts(normalize=True) * 100
class_ratio = " / ".join([f"{v:.1f}%" for v in class_counts.values])

# Пропуски считаем по ВСЕМУ датасету (как показано на твоём скриншоте)
print(f"Пропуски в признаках: {df[features].isna().values.mean()*100:.1f}% ячеек")
print(f"Пропуски во всей таблице: {df.isna().values.mean()*100:.1f}% ячеек")
print(f"Строк хотя бы с одним пропуском: {df[features].isna().any(axis=1).mean()*100:.1f}%")
print(f"Пропуски в целевой {target}: {df[target].isna().mean()*100:.1f}%")
print()
print(f"K (количество классов): {K}")
print(f"Несбалансированные классы: {class_ratio}")

Пропуски в признаках: 5.2% ячеек
Пропуски во всей таблице: 9.0% ячеек
Строк хотя бы с одним пропуском: 93.4%
Пропуски в целевой review_sentiment: 17.8%

K (количество классов): 3
Несбалансированные классы: 68.8% / 30.5% / 0.7%


## 4. Выбросы (правило 1.5·IQR)

In [19]:
# Выбросы считаем по всем числовым признакам, кроме бинарных (у которых только 2 значения)
X = df[num_cols]
X = X.loc[:, X.nunique() > 2]  # оставляем только непрерывные числовые
Q1, Q3 = X.quantile(0.25), X.quantile(0.75)
I = Q3 - Q1
out = (X < Q1 - 1.5*I) | (X > Q3 + 1.5*I)
outlier_percent = out.values.sum() / X.notna().values.sum() * 100

# Для примера смотрим на распределение customer_age
p = df["customer_age"].dropna()
q1, q3 = p.quantile([0.25, 0.75]); iqr = q3 - q1
age_out = ((p < q1 - 1.5*iqr) | (p > q3 + 1.5*iqr)).mean() * 100

print(p.describe(percentiles=[.5, .9, .99]).round(1))
print(f"\nВыбросы по customer_age: {age_out:.1f}% (граница {q3 + 1.5*iqr:.0f} лет, максимум {p.max():.0f} лет)")
print(f"Выбросы среди числовых значений: {outlier_percent:.1f}%")

count    138116.0
mean         45.9
std          16.5
min          18.0
50%          46.0
90%          69.0
99%          74.0
max          74.0
Name: customer_age, dtype: float64

Выбросы по customer_age: 0.0% (граница 102 лет, максимум 74 лет)
Выбросы среди числовых значений: 3.3%


In [20]:
print("="*40)
print("📊 СТАТИСТИКА ДЛЯ ЗАПОЛНЕНИЯ ТАБЛИЦЫ")
print("="*40)
print(f"N (число объектов): {N}")
print(f"d (число признаков): {len(features)}")
print(f"K (количество классов): {K}")
print(f"Пропущенные значения: {df[features].isna().values.mean()*100:.1f}%")
print(f"Несбалансированные классы: {class_ratio}")
print(f"Выбросы: {outlier_percent:.1f}%")
print("-"*40)
print(f"Количество числовых признаков: {len(num_cols)}")
print(f"Количество категориальных признаков: {len(cat_cols)}")
print(f"Количество текстовых признаков: {len(text_cols)}")
print(f"Количество признаков других типов: {len(other_cols)} ({other_cols[0] if other_cols else '-'})")
print("="*40)

print("\n✅ СООТВЕТСТВИЕ ТРЕБОВАНИЯМ (Галочки):")
print(f"1. Наличие пропущенных значений: {'ДА' if df[features].isna().values.mean() > 0 else 'НЕТ'}")
print(f"2. Разнородные признаки: {'ДА' if len(num_cols)>0 and len(cat_cols)>0 and len(text_cols)>0 else 'НЕТ'}")
print(f"3. Несбалансированные классы: {'ДА' if class_counts.min() < 30 else 'НЕТ'} (мин. класс: {class_counts.min():.1f}%)")
print(f"4. Большое количество выбросов: {'ДА' if outlier_percent > 5 else 'НЕТ'} (выбросов: {outlier_percent:.1f}%)")
print(f"5. Наличие текстовых признаков: {'ДА' if len(text_cols) > 0 else 'НЕТ'}")
print("="*40)

📊 СТАТИСТИКА ДЛЯ ЗАПОЛНЕНИЯ ТАБЛИЦЫ
N (число объектов): 138116
d (число признаков): 37
K (количество классов): 3
Пропущенные значения: 5.2%
Несбалансированные классы: 68.8% / 30.5% / 0.7%
Выбросы: 3.3%
----------------------------------------
Количество числовых признаков: 17
Количество категориальных признаков: 18
Количество текстовых признаков: 1
Количество признаков других типов: 1 (is_repeat_customer)

✅ СООТВЕТСТВИЕ ТРЕБОВАНИЯМ (Галочки):
1. Наличие пропущенных значений: ДА
2. Разнородные признаки: ДА
3. Несбалансированные классы: ДА (мин. класс: 0.7%)
4. Большое количество выбросов: НЕТ (выбросов: 3.3%)
5. Наличие текстовых признаков: ДА
